# **MÓDULO 27 - Projeto de Doenças Cardiovasculares - Regressão Logística**


Assim como na aula que trabalhamos com uma base de dados nova, com um contexto de modelo de propensão a compra de carros, para a atividade de vocês achei interessante trazer também novos desafios.

Nessa tarefa iremos construir um modelo que nos ajude a prever doenças cardiovasculares, a base contém dados reais.

age - idade dos pacientes

gender - genero (2 mulheres) (1 homens)

height - altura dos pacientes

weight - peso dos pacientes

gluc - glicose

smoke - fumante (1) não fumante (0)

alco - consume alcool (1) não consome (0)

active - realiza atividades fisicas (1) não realiza (0)

cardio_disease - tem doença cardio (1) não tem (0) - Variável target


Seu objetivo é utilizar esses dados históricos dos pacientes e construir um bom modelo de regressão capaz de indicar se novos pacientes estão propensos a doenças cariovasculares ou não.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, classification_report

# 1) Comece carregando e tratando a base de dados.
Assim como na aula essa nova base não passou por pré processamento nenhum então nessa etapa, carrega os dados, verifique os tipos de dados, verifique se temos dados faltantes e outliers.
Quando necessário realize o tratamento.


In [ ]:
base = pd.read_csv("CARDIO_BASE.csv", delimiter=';')

In [ ]:
base

In [ ]:
base.info()

In [ ]:
#Fazer a conversão da coluna Weight para Inteiro
base['weight'] = base['weight'].str.replace(',', '.')
base['weight'] = base['weight'].astype(float)
print(base['weight'].dtype)

In [ ]:
base.describe()

In [ ]:
base['gluc'].unique()

In [ ]:
base['cholesterol'].unique()

Colesterol e glicose tem numeros entre 1,2 e 3. Considerando que 1 está normal, 2 acima do normal, e 3 muito acima do normal

Existe uma outra questão que é com relaçao a altura e peso. Nossa base tem idade minima de 30 anos, mas a altura minima tem 70 centimetros, entao estes casos precisam ser desconsiderados, tambem casos extremos como altura com 250 centimetos. Outra questão é com relaçao ao peso, pessoas de 30 anos provavelmente não tem 30 kg, e também bem pouco provável que pesem 200 kg. vou fazer o calculo de Indice de massa coporal para ver se realmente devemos excluir estes dados com base no resultado.

In [ ]:
base['imc'] = base['weight'] / ((base['height']/100)**2)
base['imc'].describe()

In [ ]:
print(((base['imc'] < 15) | (base['imc'] > 50)).mean()*100)

In [ ]:
base[base['imc'] > 50][['age', 'height', 'weight', 'imc']].sort_values('imc', ascending=False)

In [ ]:
base[base['imc'] < 16][['age', 'height', 'weight', 'imc']].sort_values('imc', ascending=False)

Verificando os dados de imc acima de 50 e abaixo de 16. Provavelmente estes dados representam erro de digitação, como por exemplo um adulto de 2,5 metros com 86 kg, representando um imc de 13.76

Vou copiar a base com a remoção destes dados e trabalhar na base clean

In [ ]:
base_clean = base[
    (base['imc'] >=15) &
    (base['imc'] <= 50)
].copy()

In [ ]:
base_clean

In [ ]:
base_clean['cardio_disease'].value_counts(normalize=True) * 100

Dentro da base temos 50% de pessoas com problemas cardíacos e 49% sem problemas

# 2) Agora é hora de explorar os dados com uma análise bem completa.
Plote pelo menos 3 gráficos analisando o comportamento da variável cardio com outras variaveis da sua preferência (análise bivariada). Não se esqueça de trazer insights acerca do analisado.


In [ ]:
# IMC x Doença Cardíaca
sns.boxplot(
    data=base_clean,
    x='cardio_disease',
    y='imc'
)

A princípio existe um indicio de que pessoas com doenças cardíacas tem uma média de IMC maior do qeu pessoas que não tem doenças cardíacas

In [ ]:
# Idade x Doença Cardíaca
sns.boxplot(
    data=base_clean,
    x='cardio_disease',
    y='age'
)

Indicios de que pessoas mais velhas tem mais priblemas cardíacos

In [ ]:
#pessoas com colesterol alto tem mais problemas cardíacos ?
pd.crosstab(
    base_clean['cholesterol'],
    base_clean['cardio_disease'],
    normalize='index'
) * 100

A principio pessoas com colesterou alto tem amis problemas cardíacos

In [ ]:
# Glicose x Doenças Cardíacas
sns.countplot(
    data=base_clean,
    x='gluc',
    hue='cardio_disease'
)

A principio maior indice de problemas de coraçao em pessoas com alteraçao na glicemia 

In [ ]:
# fumantes x doenças cardíacas 
sns.countplot(
    data=base_clean,
    x='smoke',
    hue='cardio_disease'
)

A principio fumantes nao apresentam mais problemas de doen;cas cardíacas

# 3) Nessa etapa você deve trazer a matriz de correlação e apontar insights acerca das variáveis com um relacionamento mais forte entre si.



In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(
    base_clean.corr(numeric_only=True),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0
)

plt.title('Matriz de correlação')
plt.show()

Parece que as maiores correlacoes com problemas cardíacos estão relacionadas a IMC, idade e colesterol

# 4) Essa é a sua última etapa pré modelo. Você deve:

A) Separar a base em treino e teste.

B) Você considera que essa base precisa que os dados sejam padronizados? Se sim, porque? Se acredita que devem, então realize essa etapa.

C) Verifique se os dados estão balanceados, se não, faça o balanceamento.


D) Visualize as bases de treino, teste (X E Y) e verifique se está tudo adequado.

In [ ]:
X = base_clean.drop('cardio_disease', axis=1)
Y = base_clean['cardio_disease']
base_clean['cardio_disease'].value_counts()

In [ ]:
# Separar em base de treino e teste (usando 80% para treino e 20% para teste)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

Padronização das features é necessário pois os idade, tamanho, peso e imc tem escalas diferentes.

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
# balanceamento de dados usando o smote
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

In [ ]:
X_train_balanced

In [ ]:
y_train_balanced

In [ ]:
X_test

In [ ]:
y_test

# 5) Realize a etapa de treinamento do modelo:

A) Faça o treinamento do modelo.

B) Traga o intercept e os coeficientes.

c) Avalie as métricas do modelo treinado

D) Justifique se te parece que o modelo tem feito boas previsões ou não.

Aplicando a regressao logistica e treinando o modelo

In [ ]:
logistic_cardio = LogisticRegression(random_state = 0)

In [ ]:
logistic_cardio.fit(X_train_balanced, y_train_balanced)

In [ ]:
logistic_cardio.intercept_

In [ ]:
logistic_cardio.coef_

Valores de coeficiente atribuidos a cada variável

# 6) Teste seu modelo!

A) Aplique o modelo aos dados de teste.

B) Avalie as métricas do modelo treinado

C) Plote o gráfico da curva AUC-ROC e explique o que consegue analisar através do gráfico.

In [ ]:
# avaliaçao de classificaçao do modelo
previsoes = logistic_cardio.predict(X_train_balanced)

In [ ]:
relatorio = classification_report(y_train_balanced, previsoes)
print("Relatório de Classificação:")
print(relatorio)

In [ ]:
# aplicando treinamento na base de teste
y_pred_test = logistic_cardio.predict(X_test)


In [ ]:
relatorio = classification_report(y_test, y_pred_test)
print("Relatório de Classificação:")
print(relatorio)

Previsao para a classe um ficou em 63%, o recall sinalizo que 60% de fato foram corretamente identificadas pelo modelo e a média hamônica da precisão ficou em 61%

In [ ]:
# Curva AUC - ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_test)

roc_auc = roc_auc_score(y_test, y_pred_test)
print("AUC: {:.2f}".format(roc_auc))

In [ ]:
plt.figure()
plt.plot(fpr, tpr, color='red', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0,1], [0,1], color='black', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

# 7) Explique:

A) Explique com suas palavras regressão logistica.

B) Explique porque a regressão logistica é um modelo de classificação.

C) Explique quais pontos em comum a regressão logistica tem da regressão linear.



A - 
Regressão logística é um modelo estatístico usado para estimar a probabilidade de um evento acontecer. Em vez de prever um valor numérico contínuo, como na regressão linear, ela prevê a chance de uma observação pertencer a uma classe, por exemplo, ter ou não doença cardíaca. O resultado fica entre 0 e 1, representando uma probabilidade.

B - 
Ela é considerada um modelo de classificação porque o objetivo final é separar observações em categorias. Após calcular a probabilidade, define-se um ponto de corte, geralmente 0,5. Se a probabilidade for maior que esse valor, o modelo classifica como classe 1; se for menor, classifica como classe 0. Portanto, embora calcule probabilidades, seu uso principal é classificar.

C -
As duas utilizam uma combinação linear das variáveis explicativas para modelar a relação com a variável resposta. Em ambas, cada variável recebe um coeficiente que indica direção e intensidade do efeito sobre o resultado. Além disso, as duas são modelos supervisionados e dependem da relação entre variáveis independentes e variável alvo para fazer previsões. A principal diferença é que a regressão linear prevê valores contínuos, enquanto a regressão logística transforma o resultado em probabilidade para classificação.